# 01. Preprocessing


## Import Libraries

In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import time
import json

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

np.random.seed(42)

print("[01-PREPROCESSING] Libraries loaded")

[01-PREPROCESSING] Libraries loaded


## Configuration

In [ ]:
# Dataset path (Kaggle)
DATASET_PATH = "/kaggle/input/dataset-sampah"

# Fallback for local testing
if not os.path.exists(DATASET_PATH):
    DATASET_PATH = input("Enter dataset path: ").strip()

# Class configuration
CLASSES = ['Organik', 'Anorganik', 'Lainnya']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

# Output directory
OUTPUT_DIR = './01-preprocessing-output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[01-PREPROCESSING] Dataset path: {DATASET_PATH}")
print(f"[01-PREPROCESSING] Classes: {CLASSES}")
print(f"[01-PREPROCESSING] Output directory: {OUTPUT_DIR}")

## Scan Dataset (Collect Image Paths)

In [ ]:
def scan_dataset(dataset_path):
    """
    Scan dataset folder and collect image paths.
    TIDAK load images, hanya collect paths!
    """
    data_info = []
    class_counts = {}
    
    print("[01-PREPROCESSING] Scanning dataset folder...")
    
    for class_name in CLASSES:
        class_path = os.path.join(dataset_path, class_name)
        
        if not os.path.exists(class_path):
            print(f"[WARNING] Folder not found: {class_path}")
            continue
        
        # Supported image extensions
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
        image_files = []
        
        for ext in image_extensions:
            image_files.extend(Path(class_path).glob(f'*{ext}'))
            image_files.extend(Path(class_path).glob(f'*{ext.upper()}'))
        
        class_counts[class_name] = len(image_files)
        
        # Collect paths (TIDAK load images!)
        for img_path in image_files:
            data_info.append({
                'image_path': str(img_path),
                'class_name': class_name,
                'class_idx': CLASS_TO_IDX[class_name]
            })
    
    df = pd.DataFrame(data_info)
    
    print(f"\n[01-PREPROCESSING] Scan complete:")
    print(f"  Total images: {len(df):,}")
    for class_name, count in class_counts.items():
        percentage = count / len(df) * 100
        print(f"  {class_name}: {count:,} ({percentage:.1f}%)")
    
    return df, class_counts

# Scan dataset
start_time = time.time()
df_dataset, class_counts = scan_dataset(DATASET_PATH)
scan_time = time.time() - start_time

print(f"\n[01-PREPROCESSING] Scan time: {scan_time:.2f}s")
df_dataset.head()

## Class Distribution Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
class_names = list(class_counts.keys())
class_values = list(class_counts.values())
colors = sns.color_palette("husl", len(class_names))

bars = axes[0].bar(class_names, class_values, color=colors)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Image Count')
axes[0].set_xticklabels(class_names, rotation=45)

for bar, value in zip(bars, class_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                str(value), ha='center', va='bottom', fontweight='bold')

# Pie chart
axes[1].pie(class_values, labels=class_names, autopct='%1.1f%%', 
           colors=colors, startangle=90)
axes[1].set_title('Class Distribution (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"[01-PREPROCESSING] Visualization saved to {OUTPUT_DIR}/class_distribution.png")

## Stratified Train/Val/Test Split (70/15/15)

In [ ]:
def stratified_split(df, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    """
    Stratified split untuk memastikan proporsi class balanced.
    """
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6, "Sizes must sum to 1.0"
    
    X = df[['image_path', 'class_name']]
    y = df['class_idx']
    
    # First split: train vs (val + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, 
        test_size=(val_size + test_size), 
        stratify=y, 
        random_state=random_state
    )
    
    # Second split: val vs test
    val_ratio = val_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, 
        test_size=(1 - val_ratio),
        stratify=y_temp, 
        random_state=random_state
    )
    
    # Combine X and y
    train_df = pd.concat([X_train, y_train], axis=1)
    val_df = pd.concat([X_val, y_val], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)
    
    return train_df, val_df, test_df

print("[01-PREPROCESSING] Performing stratified split (70/15/15)...")
start_time = time.time()

train_df, val_df, test_df = stratified_split(df_dataset)

split_time = time.time() - start_time

print(f"[01-PREPROCESSING] Split completed in {split_time:.2f}s")
print(f"\nSplit Summary:")
print(f"  Train: {len(train_df):,} images ({len(train_df)/len(df_dataset)*100:.1f}%)")
print(f"  Val:   {len(val_df):,} images ({len(val_df)/len(df_dataset)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} images ({len(test_df)/len(df_dataset)*100:.1f}%)")

print("\nClass distribution per split:")
for split_name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    class_dist = split_df['class_name'].value_counts().sort_index()
    print(f"  {split_name}: {dict(class_dist)}")

## Save CSV Files (Image Paths + Labels)

In [ ]:
print(f"[01-PREPROCESSING] Saving CSV files to: {OUTPUT_DIR}")

# Save split CSV files
train_df.to_csv(f'{OUTPUT_DIR}/train_dataset.csv', index=False)
val_df.to_csv(f'{OUTPUT_DIR}/val_dataset.csv', index=False)
test_df.to_csv(f'{OUTPUT_DIR}/test_dataset.csv', index=False)

print(f"[01-PREPROCESSING] CSV files saved:")
print(f"  ✓ train_dataset.csv ({len(train_df):,} rows)")
print(f"  ✓ val_dataset.csv ({len(val_df):,} rows)")
print(f"  ✓ test_dataset.csv ({len(test_df):,} rows)")

# Save metadata
metadata = {
    'classes': CLASSES,
    'class_to_idx': CLASS_TO_IDX,
    'idx_to_class': {int(k): v for k, v in IDX_TO_CLASS.items()},
    'class_counts': class_counts,
    'total_images': len(df_dataset),
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'train_ratio': len(train_df) / len(df_dataset),
    'val_ratio': len(val_df) / len(df_dataset),
    'test_ratio': len(test_df) / len(df_dataset)
}

with open(f'{OUTPUT_DIR}/dataset_info.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"  ✓ dataset_info.json")

# List saved files
print(f"\n[01-PREPROCESSING] All files saved:")
for file in sorted(os.listdir(OUTPUT_DIR)):
    file_path = os.path.join(OUTPUT_DIR, file)
    if os.path.isfile(file_path):
        file_size = os.path.getsize(file_path) / 1024
        print(f"  {file} ({file_size:.1f} KB)")

## Preprocessing Summary Report

In [ ]:
summary = f"""
========================================
01. PREPROCESSING SUMMARY - JakOlah
========================================

Dataset Overview:
  Total images: {len(df_dataset):,}
  Classes: {len(CLASSES)} ({', '.join(CLASSES)})

Class Distribution:
"""

for class_name, count in class_counts.items():
    pct = count / len(df_dataset) * 100
    summary += f"  {class_name}: {count:,} ({pct:.1f}%)\n"

summary += f"""
Data Split (Stratified):
  Train: {len(train_df):,} images (70.0%)
  Val:   {len(val_df):,} images (15.0%)
  Test:  {len(test_df):,} images (15.0%)

Output Files:
  ✓ train_dataset.csv (image paths + labels)
  ✓ val_dataset.csv   (image paths + labels)
  ✓ test_dataset.csv  (image paths + labels)
  ✓ dataset_info.json (metadata)
  ✓ class_distribution.png (visualization)

IMPORTANT NOTES:
  1. File ini HANYA split data dan create CSV paths
  2. TIDAK ada image loading/resizing/augmentation di sini
  3. Split dilakukan SEBELUM augmentasi (mencegah data leakage)
  4. Augmentasi akan dilakukan di 02-Feature-Extraction-Step
  
Workflow:
  01 (this file): Scan folder → Split data → Save CSV paths
  02: Load images → Augment (train only) → Extract features
  03: Train SVM classifier
  04: Evaluate model

========================================
NEXT STEP: Run 02-Feature-Extraction-Step.ipynb
========================================
"""

print(summary)

with open(f'{OUTPUT_DIR}/preprocessing_summary.md', 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"[01-PREPROCESSING] Summary saved to {OUTPUT_DIR}/preprocessing_summary.md")
print(f"[01-PREPROCESSING] ✅ COMPLETED")

## Download Output

In [ ]:
import shutil

# Create zip file of all outputs
zip_filename = '01-preprocessing-output'
shutil.make_archive(zip_filename, 'zip', OUTPUT_DIR)

print(f"[01-PREPROCESSING] Output zipped to: {zip_filename}.zip")
print(f"  File size: {os.path.getsize(f'{zip_filename}.zip') / (1024*1024):.2f} MB")
print(f"\n💾 Download {zip_filename}.zip dari Kaggle output panel")